# ⚙️ Раздел 5: Конвейер за обработка на данни (Data Processing Pipeline)

За да се гарантира, че Конволюционната невронна мрежа (CNN) получава консистентни и висококачествени входни данни, е изграден автоматизиран конвейер (pipeline) за предварителна обработка. Той трансформира суровите `.wav` файлове в стандартизирани 2D тензори чрез четири последователни етапа.

### Импортиране на необходимите библиотеки

In [7]:
import os
import glob
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

### Етап 1: Екстракция и акустично пречистване (Audio Extraction & Cleaning)

Първата стъпка цели зареждането на сигнала и премахването на ненужната информация (фонов шум в началото и края).

* **Зареждане (Loading):** Аудио файловете се зареждат чрез `librosa`, като честотата на дискретизация се стандартизира (напр. 22050 Hz) за баланс между изчислителна лекота и запазване на акустичните детайли.
* **Изрязване на тишината (Trimming):** Прилага се алгоритъм за изрязване (`librosa.effects.trim`) с праг от **20dB**. Както бе установено в изследователския анализ (EDA), това гарантира, че моделът ще се фокусира върху същинското звуково събитие, а не върху предхождащата го тишина.

In [ ]:
def load_and_clean_audio(file_path, sr=22050, top_db=20):
    """
    Етап 1: Зарежда аудио файл и изрязва тишината в началото и края.
    """
    # Зареждане на аудио файла
    y, sr = librosa.load(file_path, sr=sr)
    
    # Изрязване на тишината
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    
    return y_trimmed, sr


### Етап 2: Времева стандартизация (Temporal Standardization)

Невронните мрежи изискват входни данни с фиксиран размер. Тъй като след изрязването на тишината продължителността на файловете варира, се прилага времева стандартизация до **точно 2.0 секунди**.

* **Zero-Padding (Допълване):** Ако активният звук е по-кратък от 2 секунди, сигналът се допълва с нули (тишина) в двата края, за да достигне нужния размер.
* **Cropping (Изрязване):** Ако звукът е по-дълъг, се извлича централен или произволен 2-секунден прозорец. Това запазва най-енергичната част на звука.

In [9]:
def standardize_length(y, sr, target_duration=2.0):
    """
    Етап 2: Стандартизира дължината на аудио сигнала до точно target_duration секунди.
    """
    target_length = int(target_duration * sr)
    current_length = len(y)
    
    if current_length < target_length:
        # Zero-Padding (Допълване с нули равномерно от двете страни)
        pad_length = target_length - current_length
        pad_left = pad_length // 2
        pad_right = pad_length - pad_left
        y_standardized = np.pad(y, (pad_left, pad_right), mode='constant')
    elif current_length > target_length:
        # Cropping (Извличане на централен прозорец)
        start = (current_length - target_length) // 2
        y_standardized = y[start:start + target_length]
    else:
        y_standardized = y
        
    return y_standardized

### Етап 3: Спектрална трансформация (Spectral Transformation)

Това е ядрото на конвейера, където 1D времевият сигнал се превръща в 2D визуална репрезентация (изображение), подходяща за CNN.

* **Mel-Spectrogram:** Сигналът се пропуска през 128 Мел-филтъра (`n_mels=128`), обхващащи честотния диапазон до 8000 Hz (`fmax=8000`). Този праг е избран умишлено, тъй като съдържа цялата необходима семантична информация за 50-те класа на ESC-50, елиминирайки високочестотния шум.
* **Логаритмично мащабиране (Log-Scaling):** Енергията на спектрограмата се конвертира в децибели (`power_to_db`). Това прави тихите акустични подписи (напр. стъпки) математически видими за модела, наравно със силните (напр. сирени).

In [10]:
def extract_mel_spectrogram(y, sr, n_mels=128, fmax=8000):
    """
    Етап 3: Трансформира времевия сигнал в логаритмична Мел-спектрограма.
    """
    # Изчисляване на Мел-спектрограма
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
    
    # Конвертиране на енергията в децибели (Log-Scaling)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    return mel_spec_db

### Етап 4: Форматиране за машинно обучение (ML Formatting & Normalization)

Финалният етап подготвя създадената спектрограма за директно подаване към слоевете на невронната мрежа.

* **Per-sample Нормализация (Min-Max Scaling):** Всяка индивидуална спектрограма се мащабира в диапазона **[0, 1]**. Това прави модела устойчив на вариации в силата на звука (Volume Invariance) – той ще разпознае кучето по ритъма на лая, а не по това колко близо е било до микрофона.
* **Добавяне на цветови канал (Channel Reshaping):** Тъй като CNN архитектурите очакват изображения с канали (както RGB снимките имат 3 канала), 2D матрицата се преоразмерява в 3D тензор с 1 канал (Grayscale формат).
* **Кодиране на етикетите (Label Encoding):** Текстовите категории (напр. `"siren"`, `"dog"`) се трансформират в целочислени стойности от 0 до 49, което позволява на модела да изчислява своята грешка (Loss function) по време на обучение.

In [11]:
def format_for_ml(mel_spec_db):
    """
    Етап 4: Нормализира и форматира данните за подаване към CNN.
    """
    # 1. Per-sample нормализация (Min-Max Scaling) в диапазона [0, 1]
    mel_min = mel_spec_db.min()
    mel_max = mel_spec_db.max()
    if mel_max - mel_min > 0:
        mel_normalized = (mel_spec_db - mel_min) / (mel_max - mel_min)
    else:
        mel_normalized = mel_spec_db - mel_min # запазване като нули, ако няма разлика
        
    # 2. Add channel dimension (Reshaping) 
    # От (n_mels, time_steps) към (n_mels, time_steps, 1)
    mel_tensor = mel_normalized[..., np.newaxis]
    
    return mel_tensor

# Пример за Label Encoding на масив от етикети
def encode_labels(labels):
    encoder = LabelEncoder()
    encoded_labels = encoder.fit_transform(labels)
    return encoded_labels, encoder

### Интеграция: Пълен Data Processing Pipeline

По-долу обединяваме всички четири стъпки в една обща функция, която може да се приложи върху целия набор от данни. Също така създаваме и визуализация на резултата.

In [12]:
def process_audio_file(file_path, target_duration=2.0):
    """
    Пълен конвейер за обработка на един аудио файл.
    """
    # Етап 1
    y, sr = load_and_clean_audio(file_path)
    
    # Етап 2
    y_std = standardize_length(y, sr, target_duration=target_duration)
    
    # Етап 3
    mel_spec_db = extract_mel_spectrogram(y_std, sr)
    
    # Етап 4
    mel_tensor = format_for_ml(mel_spec_db)
    
    return mel_tensor, mel_spec_db

def visualize_pipeline(file_path):
    """
    Визуализира Мел-спектрограмата след преминаване през конвейера.
    """
    try:
        mel_tensor, mel_spec_db = process_audio_file(file_path)
        
        plt.figure(figsize=(10, 4))
        # Използваме sr=22050 като стандартно от конвейера
        librosa.display.specshow(mel_spec_db, sr=22050, x_axis='time', y_axis='mel', fmax=8000)
        plt.colorbar(format='%+2.0f dB')
        plt.title('Mel-Spectrogram след Конвейера (Етап 3)')
        plt.tight_layout()
        plt.show()
        
        print(f"Форма на финалния тензор за CNN (Етап 4): {mel_tensor.shape}")
    except Exception as e:
        print(f"Възникна грешка при обработката: {e}")